In [1]:
import os
import pandas as pd
from pprint import pprint
from loguru import logger


def process_serrf_dataset(input_file: str, output_dir: str) -> None:
    """
    Parse raw Excel file, format it to pi-metaboqc standard requirements,
    and export aligned metadata and intensity DataFrames to CSVs.
    """
    logger.info(f"Loading raw data from: {input_file}")

    try:
        # Read Excel file with MultiIndex headers and indices
        int_df = pd.read_excel(
            io=input_file,
            sheet_name="Sheet1",
            header=[0, 1, 2, 3],
            index_col=[0, 1]
        )
    except Exception as e:
        logger.error(f"Failed to read data file: {e}")
        return
    
    # Remove unvalid samples
    int_df = int_df.loc[
        :,~int_df.columns.get_level_values("label").str.startswith("sample")]

    logger.info("Standardizing index structures and extracting metadata...")

    # Rename MultiIndex levels and drop the 'ID' level
    int_df = (
        int_df.rename_axis(
            columns=["Batch", "Sample Type", "Inject Order", "Sample Name"],
            index=["ID", "Metabolite"]
        ).reset_index(level="ID", drop=True)
    )

    # Extract metadata into a separate DataFrame
    meta = int_df.columns.to_frame().reset_index(drop=True)

    # Retain only the 'Sample Name' level for the intensity matrix columns
    int_df.columns = int_df.columns.get_level_values("Sample Name")

    logger.info("Formatting metadata content...")
    # Add prefix to Batch and standardize Sample Type values safely
    meta["Sample Type"] = meta["Sample Type"].replace({
        "sample": "Sample",
        "qc": "QC"})
    meta["Sample Name"] = meta.apply(
        lambda x: x["Sample Name"] if (x["Sample Type"] == "Sample") else (
            x["Batch"]+"-"+x["Sample Name"]), axis=1)
    meta["Batch"] = "Batch-" + meta["Batch"].astype(str)

    int_df.columns = meta["Sample Name"]
    return meta, int_df


def summarize_batch_info(
    meta: pd.DataFrame,
    batch_col: str = "Batch",
    order_col: str = "Inject Order",
    type_col: str = "Sample Type"
) -> pd.DataFrame:
    """
    Summarize batch details including injection order range, sample counts, 
    QC counts, and the presence of any other sample categories.
    """
    # Verify that all required columns exist in the metadata
    missing = [
        c for c in [batch_col, order_col, type_col] if c not in meta.columns
    ]
    if missing:
        raise ValueError(f"Missing required columns in meta: {missing}")

    # Ensure injection order is numeric for accurate min/max calculations
    orders = pd.to_numeric(meta[order_col], errors="coerce")
    
    summary_records = []
    
    # Iterate through each analytical batch independently
    for batch_id, group in meta.groupby(batch_col):
        # 1. Determine the injection order range safely
        b_orders = orders[group.index]
        if b_orders.notna().any():
            order_range = f"{b_orders.min():.0f} ~ {b_orders.max():.0f}"
        else:
            order_range = "Unknown"
        
        # 2. Count the occurrences of each sample type
        type_counts = group[type_col].value_counts()
        sample_count = type_counts.get("Sample", 0)
        qc_count = type_counts.get("QC", 0)
        
        # Append the calculated metrics as a dictionary record
        summary_records.append({
            "Batch ID": batch_id,
            "Injection Order": order_range,
            "Samples": sample_count,
            "Pooled QCs": qc_count
        })
        
    # Convert the records list to a structured DataFrame
    summary_df = pd.DataFrame(summary_records)
    
    return summary_df


# if __name__ == "__main__":
# Define directories safely across different operating systems
input_dir = os.path.abspath(
    os.path.join("..", "..", "data", "raw", "SERRF")
)
output_dir = os.path.abspath(
    os.path.join("..", "..", "data", "processed", "SERRF")
)

data_file  = os.path.join(input_dir, "SERRF example dataset.xlsx")


# Execute the preprocessing pipeline
metadata_df, intensity_df = process_serrf_dataset(
    input_file=data_file, output_dir=output_dir)
pprint(summarize_batch_info(meta=metadata_df))

# Ensure output directory exists safely
os.makedirs(output_dir, exist_ok=True)

# Define output file paths
meta_path = os.path.join(output_dir, "project_meta.csv")
int_path = os.path.join(output_dir, "project_intensity.csv")

logger.info("Saving processed datasets...")

# Save the properly assigned meta variable
metadata_df.to_csv(
    path_or_buf=meta_path,
    index=False,
    header=True,
    encoding="utf-8-sig"
)
logger.success(f"Metadata saved successfully in: {meta_path}")

# Save the properly assigned int_df variable
intensity_df.to_csv(
    path_or_buf=int_path,
    encoding="utf-8-sig",
    na_rep="N/A"
)
logger.success(f"Intensity data saved successfully in: {int_path}")

2026-05-27 23:12:06.236 | INFO     | __main__:process_serrf_dataset:12 - Loading raw data from: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\SERRF\SERRF example dataset.xlsx
2026-05-27 23:12:08.076 | INFO     | __main__:process_serrf_dataset:30 - Standardizing index structures and extracting metadata...
2026-05-27 23:12:08.080 | INFO     | __main__:process_serrf_dataset:46 - Formatting metadata content...
2026-05-27 23:12:08.090 | INFO     | __main__:<module>:134 - Saving processed datasets...
2026-05-27 23:12:08.093 | SUCCESS  | __main__:<module>:143 - Metadata saved successfully in: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\SERRF\project_meta.csv
2026-05-27 23:12:08.193 | SUCCESS  | __main__:<module>:151 - Intensity data saved successfully in: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\SERRF\project_intensity.csv


  Batch ID Injection Order  Samples  Pooled QCs
0  Batch-A         1 ~ 335      300          32
1  Batch-B       336 ~ 671      300          33
2  Batch-C      672 ~ 1006      300          32
3  Batch-D     1007 ~ 1299      262          28


In [2]:
pprint(metadata_df.head(5))

     Batch Sample Type  Inject Order Sample Name
0  Batch-A          QC             1     A-QC000
1  Batch-A      Sample             3    GB001617
2  Batch-A      Sample             4    GB001333
3  Batch-A      Sample             5    GB001191
4  Batch-A      Sample             6    GB001827


In [3]:
pprint(intensity_df.shape)
pprint(intensity_df.head(5))

(268, 1287)
Sample Name                              A-QC000  GB001617  GB001333  \
Metabolite                                                             
1_ISTD Ceramide (d18:1/17:0) [M+HCOO]-    167879    158256    164492   
1_ISTD CUDA [M-H]-                         75578     76082     74334   
1_ISTD FA (16:0)-d3 [M-H]-                 71916     66125     68269   
1_ISTD LPC (17:0) [M+HCOO]-                43222     36637     42339   
1_ISTD LPE (17:1) [M-H]-                   33727     29950     32090   

Sample Name                              GB001191  GB001827  GB001722  \
Metabolite                                                              
1_ISTD Ceramide (d18:1/17:0) [M+HCOO]-     155000    150957    134195   
1_ISTD CUDA [M-H]-                          74702     72993     69064   
1_ISTD FA (16:0)-d3 [M-H]-                  64161     61638     56286   
1_ISTD LPC (17:0) [M+HCOO]-                 38337     35994     31553   
1_ISTD LPE (17:1) [M-H]-                    3